# Исследование поискового запроса

Одна Hydra-композиция позволяет посмотреть разбор и DSL, выполнить поиск, прогнать CSV `id,text` и сохранить выдачи.

Откройте notebook в Jupyter/VS Code с Python 3.12+ kernel проекта. Jupyter установите отдельно. Рабочим каталогом может быть корень checkout или вложенный каталог.

Укажите URL, индекс и поля своего источника. Authorization берётся из `OPENSEARCH_AUTHORIZATION` окружения kernel. Не вставляйте credentials в ячейки или overrides и не выводите resolved config. Результаты содержат документы индекса; поставляемый notebook сохранён без outputs.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs/main.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.core.transformers.data import PipelineData
from src.domain.request.filters import QueryFilter

## Параметры

Замените пример mapping схемой своего индекса. Для одного text-поля укажите его в `fields`, а `title_field` оставьте `null`. `source_overrides` принимает overrides словарей, parsing/retrieval/graph groups, весов и глубины. Для fuzzy нужны graph `fuzzy` и pipeline `fuzzy`.

`pipeline` выбирает полный путь поиска. Пути отсчитываются от checkout; для своих данных можно указать абсолютный путь. Выходной файл не перезаписывается.


In [ ]:
source_overrides = [
    "connection.url=http://localhost:9200",
    "jobboard.source.index=vacancies",
    "jobboard.source.fields=[title,description]",
    "jobboard.source.title_field=title",
]
pipeline = "parsed"  # baseline, lexical, parsed, fused; fuzzy с graph/fuzzy
text = "python backend developer"
input_path = ROOT / "examples/queries.csv"
output_path = ROOT / "outputs" / "notebook" / "searches.csv"

## Сборка эксперимента

Compose используется как context manager, поэтому ячейку можно повторять. Exporters отключены: Compose API не требует CLI runtime Hydra и не пишет файлы автоматически.
После изменения параметров повторите сборку; существующие объекты сохраняют прежние настройки.


In [ ]:
with initialize_config_dir(config_dir=str(ROOT / "configs"), version_base="1.3"):
    cfg = compose(
        config_name="main",
        overrides=[
            "~experiments/vacancy-search/interactive@experiment",
            "+experiments/vacancy-search/jobboard/lexical@experiment=001-interactive",
            *source_overrides,
            f"experiment.pipeline={pipeline}",
            "experiment.exporters=[]",
        ],
    )
experiment = instantiate(cfg.experiment, _convert_="all")

## Разбор и DSL без HTTP

Выберите поисковый лист: `parsed_leaf`, `lexical_leaf` или `baseline_leaf` без parser. Fused выполняет lexical и parsed; их запросы можно посмотреть отдельно. `build_query` использует тот же compiler, что и поиск.

Payload включает context и статические filters конфига. Дополнительные условия ручного поиска включайте и в payload просмотра.


In [ ]:
leaf = experiment.registry["parsed_leaf"]
parsed = leaf.parser.parse(text)
probe = PipelineData(
    context=instantiate(cfg.experiment.context, _convert_="all"),
    filters=[*instantiate(cfg.experiment.filters, _convert_="all"), QueryFilter(text)],
)
dsl = leaf.build_query(probe)
print(parsed)
print(json.dumps(dsl, ensure_ascii=False, indent=2))

## Поиск

Ячейка обращается к OpenSearch и возвращает независимый снимок SERP. Вызов можно повторять, в том числе после ошибки.
`search(text, context={...}, filters=[...])` переопределяет свойства context верхнего уровня и добавляет filters к статическим условиям эксперимента.


In [ ]:
serp = experiment.search(text)
rows = [
    {"id": item.id, "score": item.score, "fields": item.fields} for item in serp.items
]
rows

## Batch без разметки

CSV содержит ровно колонки `id,text`: id уникальны, оба значения непустые. Весь файл проверяется до первого запроса, поиск выполняется последовательно. `run()` возвращает снимок результатов без записи.

Используется собранный `experiment` с его pipeline, context, filters и параметрами. Настройки batch также задаются Hydra. CLI-конфиг `003-batch` описан в [руководстве](../../../docs/usage.md#csv-batch).


In [ ]:
batch_cfg = OmegaConf.create(
    {
        "_target_": "src.experiments.batch.search.BatchSearchExperiment",
        "input_path": str(input_path),
        "exporters": [
            {
                "_target_": "src.components.reporting.exporters.csv.SearchCSVExporter",
                "path": str(output_path),
            }
        ],
    }
)
batch = instantiate(batch_cfg, search=experiment, _convert_="all")
artifact = batch.run()
artifact.to_rows()[:5]

## Сохранение CSV

Одна строка на документ: `query_id,query,rank,document_id,score,fields`. `fields` содержит полный JSON. Для пустой выдачи записывается строка с пустыми полями результата, для null score — пустая ячейка.

Существующий файл сохраняется. Для следующего опыта выберите новый путь и пересоберите batch. Ошибка поиска прекращает запуск без частичного CSV.


In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
for exporter in batch.exporters:
    exporter.export(artifact)
output_path